<a href="https://colab.research.google.com/github/embark-cybertraining/embark-scratch-notebooks/blob/main/session_3/session_3_EDI_OBIS_map_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prototype notebook for EMBARK CyberTraining Session 3

## Use Case Goal:

An oceanographer would like to find, access, and explore data from long-term research programs in a named marine region, focusing on particulate organic matter and occurrences of selected marine species in the California Current Large Marine Ecosystem.

Data repositories: Environmental Data Initiative (EDI), Ocean Biodiversity Information System (OBIS), Marine Regions

Tools include: EDI REST API (note: requires authentication to access data), OBIS API, Python software libraries (e.g., pandas, requests, folium)
This notebook may be run locally, or within cloud-based computing infrastructure (e.g., Google Colab, OOI JupyterHub).

This notebook may be run locally, or within cloud-based computing infrastructure (e.g., Google Colab, OOI JupyterHub).  It is intended for learning purposes, aiming to inspire you to try cyberinfrastructure (CI) tools, resources, and data in your workflows.

## Credits for this notebook
* Overall the notebook was developed by the EMBARK CyberTraining project team, including Amber York, and Stace Beaulieu.
* Where indicated, some code was generated or improved by AI coding assistance, including ChatGPT, Claude, and Google Gemini.

# Part A - Dataset: EDI

## Setup for Part A

For coded workflows, EDI recommends using an API access key for authentication when using the repository REST API.
* https://edirepository.org/user-guides/working-with-access-keys

In [1]:
# Install packages used in Part A (python-dotenv reads the EDI key from a .env file)
%pip install -q python-dotenv folium

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Setup/imports for Part A
import os
import pandas as pd
import folium
from dotenv import load_dotenv

## Step 1 - Discover - EDI Dataset Summary

This dataset contains particulate organic matter (POM) measurements collected by the California Current Ecosystem Long-Term Ecological Research (CCE-LTER) program.

The notebook uses:
- sampling coordinates
- sampling depth
- date/time
- cruise and station metadata
- particulate carbon and nitrogen measurements

These data provide environmental and biogeochemical context for comparison with marine biodiversity occurrence records.

**Repository:** Environmental Data Initiative (EDI)  
**Dataset landing page:**  
https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13

**Direct data access endpoint used in notebook:**  
https://pasta.lternet.edu/package/data/eml/knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938

## Step 2 - Access - EDI URL

strategy: Load from a file using URL and access key

In [3]:
# Access EDI data
#
# Strategy:
#   1. Load the EDI access key from a .env file (JupyterLab or another notebook environment, primary method).
#   2. If no key is found, try a Colab secret named EDI_ACCESS_KEY (secondary method).
#   3. If neither is available, stop with an error, since the data cannot be accessed without a key.
#      See EDI snippet for python access to our dataset of interest:
#       https://portal.edirepository.org/nis/codeGeneration?packageId=knb-lter-cce.104.13&statisticalFileType=py
#
# Requirements:
#   * JupyterLab (or any environment where you can create a .env file (e.g. VS Code)):
#       - A .env file containing EDI_ACCESS_KEY=your_key_here  (can be in notebook folder or higher up)
#           (have .env to .gitignore so the key is never committed).
#       - The python-dotenv package: %pip install -qq python-dotenv
#   * Colab: a Secret named EDI_ACCESS_KEY with Notebook access turned on.

import os
import pandas as pd
from dotenv import load_dotenv

# Base URL of the EDI data entity to download
base_url = (
    "https://pasta.lternet.edu/package/data/eml/"
    "knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938"
)

# Load variables from a .env file (defaults to the current directory, otherwise pass a path)
load_dotenv()

# Read the access key from the environment (None if not set)
myEDIAccessKey = os.environ.get("EDI_ACCESS_KEY")

# If no .env value was found, try a Colab secret
if not myEDIAccessKey:
    try:
        from google.colab import userdata
        myEDIAccessKey = userdata.get("EDI_ACCESS_KEY")
    except Exception:  # Not in Colab, or the secret is missing or not shared with the notebook
        pass

# Stop with a clear message if no key was found by either method
if not myEDIAccessKey:
    raise RuntimeError('''No EDI access key found. Add EDI_ACCESS_KEY to a .env file or a Colab secret 
    (depending on which you are using).''')
print("Using EDI access key")

# Download the data into a DataFrame and report the record count
df_edi = pd.read_csv(f"{base_url}?key={myEDIAccessKey}")
print(f"EDI records retrieved: {len(df_edi)}")

# Display 5 random rows
df_edi.sample(5)

Using EDI access key
EDI records retrieved: 3295


,studyName,Datetime UTC,Transect,Station,Latitude (º),Longitude (º),R2R Event,CCE Event,Cast Number,Sample Type,Bottle Number,Depth (m),N (µg/L),C (µg/L),N (µmol/L),C (µmol/L),C/N Molar Ratio,Notes,Cruise,Associative Bottle Number
1597,P1408MV,2014-08-23 09:48:23,CYCLE 3,NaN,34.41410,-121.2787,NaN,340.0,35,NaN,2.0,60.0,13.844907,53.072526,0.988446,4.418770,4.470422,NaN,CCE-P1408,NaN
64,P0605KN,2006-05-20 08:04:07,CYCLE 2,NaN,33.49394,-122.0862,NaN,370.0,31,NaN,23.0,5.0,9.819563,26.024395,0.701060,2.166768,3.090703,NaN,CCE-P0605,NaN
1593,P1408MV,2014-08-23 09:48:23,CYCLE 3,NaN,34.41410,-121.2787,NaN,340.0,35,NaN,15.0,12.0,14.083399,66.173253,1.005473,5.509525,5.479536,NaN,CCE-P1408,NaN
1583,P1408MV,2014-08-22 18:19:37,CYCLE 3,NaN,34.40119,-121.3424,NaN,324.0,33,NaN,15.0,9.0,17.281951,65.494294,1.233831,5.452996,4.419564,NaN,CCE-P1408,NaN
1409,P1208MV,2012-08-21 05:27:36,EFRONT2,NaN,34.57217,-122.7134,NaN,522.0,60,NaN,4.0,39.0,18.752018,95.970996,1.338785,7.990458,5.968439,NaN,CCE-P1208,NaN


In [4]:
df_edi.info()

<class 'pandas.DataFrame'>
RangeIndex: 3295 entries, 0 to 3294
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   studyName                  3295 non-null   str    
 1   Datetime UTC               3295 non-null   str    
 2   Transect                   3199 non-null   str    
 3   Station                    468 non-null    str    
 4   Latitude (º)               3278 non-null   float64
 5   Longitude (º)              3278 non-null   float64
 6   R2R Event                  468 non-null    str    
 7   CCE Event                  3081 non-null   float64
 8   Cast Number                3295 non-null   int64  
 9   Sample Type                468 non-null    str    
 10  Bottle Number              3238 non-null   float64
 11  Depth (m)                  3238 non-null   float64
 12  N (µg/L)                   3295 non-null   float64
 13  C (µg/L)                   3295 non-null   float64
 14  N (

## Step 3 - Wrangle - Quality Control and Cleaning

In [5]:
# Clean EDI data
# Remove rows with no lat/lon
# Remove rows with missing varibles of interest (particulate C or N values)

# variables of interest
vars_of_interest = [
    "N (µg/L)",
    "C (µg/L)"
]

print(f"EDI records retrieved (total): {len(df_edi)}")

df_edi[["Latitude (º)", "Longitude (º)"]] = (
    df_edi[["Latitude (º)", "Longitude (º)"]]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows removed - missing [lat,lon]:",
      df_edi[["Latitude (º)", "Longitude (º)"]]
      .isna().any(axis=1).sum())

df_edi = df_edi.dropna(
    subset=["Latitude (º)", "Longitude (º)"]
)
# ---- Vars of interest

print("Rows removed - missing "+str(vars_of_interest)+":",
      df_edi[vars_of_interest]
      .isna()
      .all(axis=1)
      .sum()
    )

# Drop rows where all target variables are missing
df_edi = df_edi.dropna(
        subset=vars_of_interest,
        how="all"
    )

print(f"Rows remaining: {len(df_edi)}")

EDI records retrieved (total): 3295
Rows removed - missing [lat,lon]: 17
Rows removed - missing ['N (µg/L)', 'C (µg/L)']: 0
Rows remaining: 3278


In [6]:
# Inspect EDI data

print("\nDatetime range:")
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").min())
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").max())

print("\nDepth range:")
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").min())
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").max())

## todo: add min/max of variables particulate C&N of interest


Datetime range:
2006-05-11 09:40:48
2024-03-17 10:08:00

Depth range:
1.0
801.0


## Step 4 - Reuse - Visualize with Map


In [7]:
# Create reusable feature group for EDI points
edi_popup_cols = df_edi.columns.tolist()

edi_points = folium.FeatureGroup(name="EDI Points")

# this section formats information to show in a popup when you click the point
for _, row in df_edi.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in edi_popup_cols
        if pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["Latitude (º)"], row["Longitude (º)"]],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(edi_points)


# make a map
m_cc = folium.Map(
    location=[37, -123],
    zoom_start=5,
    tiles="OpenStreetMap" #or try "CyclOSM" or "Esri.WorldStreetMap"
)

# add points
edi_points.add_to(m_cc)

m_cc

# Part B - Dataset: OBIS

## Setup for Part B

In [8]:
# Install packages used in Part B
%pip install -q folium

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Setup/imports for Part B
import requests
import pandas as pd
import folium
from IPython.display import display, HTML  # display for tables, HTML for clickable links

SCI_NAME = 'Pyrosoma'  # second run will use 'Thaliacea'

# OBIS area_id for the California Current (Marine Regions MRGID 8549)
AREA_ID = 40003

# Record limit for the OBIS request; checked after retrieval to confirm all records were returned
# See https://obis.org/data/access/ for small vs large request strategy
RECORD_LIMIT = 10000

## Step 1 - Discover - OBIS Dataset Summary

**Repository:** Ocean Biodiversity Information System (OBIS)  
**API endpoint used in notebook:**  
https://api.obis.org/v3/occurrence

**OBIS area reference:**  
https://api.obis.org/v3/area

**OBIS source information:**  
https://obis.org/sources/

### OBIS Query Summary

The notebook queries OBIS occurrence records for:

- scientific name: *Pyrosoma atlanticum*
- region: California Current Large Marine Ecosystem

## Step 2 - Access - OBIS API

Strategy: Access data using the OBIS API directly (not requiring an additional library).

In [10]:
# Access OBIS data as DataFrame

response = requests.get(
    "https://api.obis.org/v3/occurrence",
    params={
        "scientificname": SCI_NAME,
        "areaid": AREA_ID,
        "size": RECORD_LIMIT,
    },
)

df_obis = pd.DataFrame(response.json()["results"])

print(f"OBIS records retrieved: {len(df_obis)}")

if len(df_obis) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



OBIS records retrieved: 1870
All matching records retrieved.


In [11]:
df_obis.info()

<class 'pandas.DataFrame'>
RangeIndex: 1870 entries, 0 to 1869
Columns: 131 entries, basisOfRecord to county
dtypes: bool(3), float64(15), int64(11), object(3), str(99)
memory usage: 3.5+ MB


In [12]:
df_obis.columns.to_list()

['basisOfRecord',
 'catalogNumber',
 'class',
 'classid',
 'collectionCode',
 'coordinateUncertaintyInMeters',
 'countryCode',
 'datasetName',
 'dateIdentified',
 'date_end',
 'date_mid',
 'date_start',
 'date_year',
 'decimalLatitude',
 'decimalLongitude',
 'eventDate',
 'eventTime',
 'family',
 'familyid',
 'genus',
 'genusid',
 'geodeticDatum',
 'identificationID',
 'identifiedBy',
 'institutionCode',
 'kingdom',
 'kingdomid',
 'license',
 'marine',
 'modified',
 'occurrenceID',
 'occurrenceStatus',
 'order',
 'orderid',
 'phylum',
 'phylumid',
 'recordedBy',
 'references',
 'rightsHolder',
 'scientificName',
 'scientificNameID',
 'species',
 'speciesid',
 'stateProvince',
 'subfamily',
 'subfamilyid',
 'subphylum',
 'subphylumid',
 'superdomain',
 'superdomainid',
 'taxonID',
 'taxonRank',
 'verbatimEventDate',
 'verbatimLocality',
 'id',
 'dataset_id',
 'node_id',
 'dropped',
 'absence',
 'originalScientificName',
 'aphiaID',
 'flags',
 'bathymetry',
 'shoredistance',
 'sst',
 'ss

## Step 3 - Wrangle - Quality Control and Cleaning

In [13]:
# Clean, inspect, and summarize OBIS data

#Not all records show the datasetName or datasetID but that can be obtained
# from the obis identifier dataset_id using another query
def get_obis_dataset_title(dataset_id):
    try:
        r = requests.get(
            "https://api.obis.org/dataset",
            params={"datasetid": dataset_id},
            timeout=30,
        )
        r.raise_for_status()

        data = r.json()

        if isinstance(data, dict):
            records = data.get("results", data.get("data", []))
        else:
            records = data

        if records:
            return records[0].get("title", "")

    except Exception:
        pass

    return ""


# Add dataset titles from OBIS dataset API
dataset_titles = {
    dataset_id: get_obis_dataset_title(dataset_id)
    for dataset_id in df_obis["dataset_id"].dropna().unique()
}

df_obis["datasetTitle"] = df_obis["dataset_id"].map(dataset_titles)


df_obis["decimalLatitude"] = pd.to_numeric(
    df_obis["decimalLatitude"],
    errors="coerce"
)

df_obis["decimalLongitude"] = pd.to_numeric(
    df_obis["decimalLongitude"],
    errors="coerce"
)

# We only want records where there are lat/lons
df_obis = df_obis.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).copy()

preview_cols = [ #used for displaying df and in map popup when clicking point
    "scientificName",
    "scientificNameID",  # LSID
    "aphiaID",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "minimumDepthInMeters",
    "maximumDepthInMeters",
    "occurrenceStatus",
    "datasetTitle",
    "occurrenceID",
]

preview_cols = [c for c in preview_cols if c in df_obis.columns]

# Display what we got
print("\nPreview of key fields (random 10 rows):")
display(df_obis[preview_cols].sample(10))




Preview of key fields (random 10 rows):


,scientificName,scientificNameID,aphiaID,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,occurrenceStatus,datasetTitle,occurrenceID
1777,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.841700,-121.990000,1991-05-15T04:10:00Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,3345a11d-0764-4409-93e6-5ff4052a4e61
593,Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,36.583858,-122.516411,2000-11-14T21:22:00Z,221.38,221.38,NaN,Video Annotation and Reference System (VARS) d...,NaN
1400,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.406870,-119.842727,2022-07-15T11:23:43-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/12638...
998,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.623912,-121.940250,2025-01-12T16:12:00-08:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/25844...
1128,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.407323,-119.842423,2022-07-15T11:25:50-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/12638...
1241,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.404926,-119.845051,2022-07-18T18:07:26-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/12691...
1009,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.462392,-120.033294,2021-04-17T17:28:34-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/74314871
1226,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.594967,-121.961668,2025-05-19T15:18:23-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/28253...
673,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,45.834536,-123.968481,2024-12-09T16:29:21-08:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/25472...
213,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.555000,-122.170000,2000-06-01T05:12:00Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,79f00482-c56a-480e-817f-dbab3b2869c7


In [14]:
# Further Inspect data (data sources, ids, etc)

# Unique species with WoRMS page links
species_list_df = (
    df_obis[["scientificName", "scientificNameID", "aphiaID"]]
    .drop_duplicates()
    .sort_values(by="scientificName")
    .reset_index(drop=True)
)

species_list_df["WoRMS_link"] = species_list_df["aphiaID"].apply(
    lambda x: (
        f'<a href="https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}" '
        f'target="_blank">WoRMS {int(x)}</a>'
    )
    if pd.notna(x)
    else ""
)

# Count occurrenceStatus values with associated scientific names

occurrence_status_summary = (
    df_obis.groupby("occurrenceStatus")["scientificName"]
    .agg(
        count="size",
        unique_scientificNames=lambda x: sorted(x.dropna().unique())
    )
    .reset_index()
)

# Show unique occurrenceStatus values for each scientificName

species_occ_status = (
    df_obis[
        ["scientificName", "occurrenceStatus"]
    ]
    .fillna("Not Supplied")
    .value_counts()
    .reset_index(name="count")
    .reset_index(drop=True)
)

# Unique datasets with clickable OBIS dataset links
df_obis["datasetURL"] = (
    "https://obis.org/dataset/" + df_obis["dataset_id"].astype(str)
)

datasets = (
    df_obis[["dataset_id", "datasetTitle", "datasetURL"]]
    .drop_duplicates()
    .sort_values(by="datasetTitle")
    .reset_index(drop=True)
)

# Make datasetURL clickable
datasets["datasetURL"] = datasets["datasetURL"].apply(
    lambda x: f'<a href="{x}" target="_blank">{x}</a>'
)


# Display more info

print(f"\nTotal unique datasets: {len(datasets)}")
display(HTML(datasets.to_html(escape=False, index=False)))

print("Unique species in the dataset:")
display(HTML(species_list_df.to_html(escape=False, index=False)))

print("Unique occurrenceStatus values per scientificName:")
display(species_occ_status)

print("\nDataFrame cols")
print(df_obis.columns.tolist())



Total unique datasets: 8


dataset_id,datasetTitle,datasetURL
aa16d305-d413-4c4a-90be-b1ec3298d58d,CAS Invertebrate Zoology (IZ),https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d
92af50ad-c9d7-4fe7-b012-eae0e77233a2,Los Angeles Urban Ocean Expedition 2019,https://obis.org/dataset/92af50ad-c9d7-4fe7-b012-eae0e77233a2
71c2c816-7e94-40b9-8e28-8172d9c5fefb,NOAA Ocean Exploration seawater eDNA metabarcoding (EX2301),https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb
89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrate Zoology Collections,https://obis.org/dataset/89e23fc8-3f61-4480-9de3-358fe6eefe0b
5b6251f6-a7a5-4dc9-994d-c9504b54776f,"Rockfish Recruitment and Ecosystem Assessment Survey, Catch Data",https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
c5687a17-e454-40f9-9a4b-d04b2c812d74,UF Invertebrate Zoology,https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74
a419c8da-35ed-4b62-9709-39b56369c44e,Video Annotation and Reference System (VARS) database,https://obis.org/dataset/a419c8da-35ed-4b62-9709-39b56369c44e
eaea291a-1e1d-4382-b86f-ac3cc15b8d5a,iNaturalist Research-grade Observations Marine Subset,https://obis.org/dataset/eaea291a-1e1d-4382-b86f-ac3cc15b8d5a


Unique species in the dataset:


scientificName,scientificNameID,aphiaID,WoRMS_link
Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,WoRMS 137224
Pyrosoma,NaN,137224,WoRMS 137224
Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,WoRMS 137250
Pyrosoma atlanticum,NaN,137250,WoRMS 137250


Unique occurrenceStatus values per scientificName:


,scientificName,occurrenceStatus,count
0,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,1260
1,Pyrosoma atlanticum,present,477
2,Pyrosoma,Not Supplied,127
3,Pyrosoma atlanticum,Not Supplied,6



DataFrame cols
['basisOfRecord', 'catalogNumber', 'class', 'classid', 'collectionCode', 'coordinateUncertaintyInMeters', 'countryCode', 'datasetName', 'dateIdentified', 'date_end', 'date_mid', 'date_start', 'date_year', 'decimalLatitude', 'decimalLongitude', 'eventDate', 'eventTime', 'family', 'familyid', 'genus', 'genusid', 'geodeticDatum', 'identificationID', 'identifiedBy', 'institutionCode', 'kingdom', 'kingdomid', 'license', 'marine', 'modified', 'occurrenceID', 'occurrenceStatus', 'order', 'orderid', 'phylum', 'phylumid', 'recordedBy', 'references', 'rightsHolder', 'scientificName', 'scientificNameID', 'species', 'speciesid', 'stateProvince', 'subfamily', 'subfamilyid', 'subphylum', 'subphylumid', 'superdomain', 'superdomainid', 'taxonID', 'taxonRank', 'verbatimEventDate', 'verbatimLocality', 'id', 'dataset_id', 'node_id', 'dropped', 'absence', 'originalScientificName', 'aphiaID', 'flags', 'bathymetry', 'shoredistance', 'sst', 'sss', 'country', 'datasetID', 'depth', 'eventID', '

## Step 4 - Reuse - Map EDI and OBIS data together

Plot points from EDI and OBIS data together on one map.

In [15]:
# Use only preview columns in popup
obis_popup_cols = preview_cols

# Create reusable feature group for OBIS points
obis_points = folium.FeatureGroup(name="OBIS Points")

#this section formats information to show in popups when you click a point
for _, row in df_obis.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in obis_popup_cols
        if col in df_obis.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(obis_points)


In [16]:
# Map EDI and OBIS data together

#map m_cc was already created in the EDI data section.

# add new OBIS points to the map that already had EDI points
obis_points.add_to(m_cc)

m_cc

# Additional Access Strategy: pyobis library

Load data from a bounding box using the pyobis library.  We will calculate the bounding box of the combined EDI and OBIS data. Then we will use that bounding box in a new query to get OBIS data using the python libary pyobis.

### Setup for pyobis and geospatial

In [17]:
# Setup/imports for the Additional Access Strategy sections (requires Parts A and B to be run first)
%pip install -q pyobis shapely

import os, re
from shapely.geometry import box
from pyobis import occurrences

# Marine Regions MRGID for the California Current (corresponds to OBIS area_id 40003)
MRGID = 8549

Note: you may need to restart the kernel to use updated packages.


## Calculate combined data bounding box

In [18]:
# Get data bounding box (WKT) from superset of EDI and OBIS data
# we did this above, but this cell will need
#   from shapely.geometry import box
#   import pandas as pd

# Combine lat/lon from both datasets and calculate one bounding box directly
combined_lats = pd.concat([df_obis["decimalLatitude"], df_edi["Latitude (º)"]])
combined_lons = pd.concat([df_obis["decimalLongitude"], df_edi["Longitude (º)"]])

minx, maxx = combined_lons.min(), combined_lons.max()
miny, maxy = combined_lats.min(), combined_lats.max()

# Build the combined bounding box polygon and WKT
combined_bbox_polygon = box(minx, miny, maxx, maxy)
combined_bbox_wkt = combined_bbox_polygon.wkt

print("Combined bounding box WKT:")
print(combined_bbox_wkt)

Combined bounding box WKT:
POLYGON ((-116.932998657 29.8670005798, -116.932998657 46.5919961817, -130.547487 46.5919961817, -130.547487 29.8670005798, -116.932998657 29.8670005798))


In [ ]:
# Create map centered on combined bounding box
m_combined = folium.Map(
    location=[
        (combined_bbox_polygon.bounds[1] + combined_bbox_polygon.bounds[3]) / 2,
        (combined_bbox_polygon.bounds[0] + combined_bbox_polygon.bounds[2]) / 2
    ],
    zoom_start=5,
    tiles="OpenStreetMap" #or try "CyclOSM" or "Esri.WorldStreetMap"
)

# Add points from before
edi_points.add_to(m_combined)
obis_points.add_to(m_combined)


# Add combined bounding box with WKT popup
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(combined_bbox_polygon.wkt, max_width=500)
).add_to(m_combined)

# Add layer control
folium.LayerControl().add_to(m_combined)

# Add popup showing latitude/longitude anywhere clicked on the map
m_combined.add_child(folium.LatLngPopup())

m_combined

### Looking at expanded BB for OBIS data

query using pyobis and the polyon of the previous obis query which corresponded to data from  area 40003 plus edi dataset bounds

In [20]:
# Access OBIS data using pyobis and the combined bounding box
#   * We imported at the top of our notebook, but this cell needs:
#     from pyobis import occurrences

response = occurrences.search(
    scientificname=SCI_NAME,
    geometry=combined_bbox_wkt, #calculated from our edi and obis data
    size=RECORD_LIMIT
)

df_obis_expanded = response.execute()


print(f"OBIS records retrieved: {len(df_obis_expanded)}")

if len(df_obis_expanded) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



2026-11-05 19:36:32 - pyobis.cache.cache - INFO - Cache initialized at /home/jovyan/.cache/pyobis
2026-11-05 19:36:32 - pyobis.obisutils - INFO - 10000 to be fetched. Estimated time =0.025572777986526491 seconds
2026-11-05 19:36:32 - pyobis.obisutils - INFO - Fetching: [███████████████████████████████████████████████████████████████████████████████████████████████████] 10000/10000
2026-11-05 19:36:32 - pyobis.cache.cache - INFO - Cache initialized at /home/jovyan/.cache/pyobis
2026-11-05 19:36:33 - pyobis.obisutils - INFO - Fetching: [████████████████████████████████████████████████████████████████████████████████████████████████████] 10000/10000
2026-11-05 19:36:33 - pyobis.cache.cache - INFO - Cache initialized at /home/jovyan/.cache/pyobis
2026-11-05 19:36:33 - pyobis.obisutils - INFO - Fetched 10000 records.


OBIS records retrieved: 2040
All matching records retrieved.


In [21]:
# Find OBIS expanded records that were not present in the original dataset

# Use occurrenceID if available and unique
new_obis_records = df_obis_expanded[
    ~df_obis_expanded["occurrenceID"].isin(df_obis["occurrenceID"])
].copy()

print(f"New OBIS records found: {len(new_obis_records)}")

# Preview new records
new_obis_records.head()

New OBIS records found: 168


,basisOfRecord,catalogNumber,class,classid,collectionCode,coordinateUncertaintyInMeters,countryCode,datasetName,dateIdentified,date_end,...,verbatimLongitude,sex,georeferenceSources,otherCatalogNumbers,verbatimCoordinates,footprintSRS,specificEpithet,unaccepted,county,identificationVerificationStatus
16,HumanObservation,257990362,Thaliacea,22626,Observations,NaN,US,iNaturalist research-grade observations,2025-01-10T00:51:06Z,1.724717e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,HumanObservation,90695778,Thaliacea,22626,Observations,5.0,US,iNaturalist research-grade observations,2021-08-11T21:43:06Z,1.628467e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,HumanObservation,30262550,Thaliacea,22626,Observations,20.0,US,iNaturalist research-grade observations,2019-08-06T04:11:34Z,1.564963e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,HumanObservation,55096185,Thaliacea,22626,Observations,5.0,US,iNaturalist research-grade observations,2020-08-01T23:33:25Z,1.595981e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,HumanObservation,84722984,Thaliacea,22626,Observations,6.0,US,iNaturalist research-grade observations,2021-06-27T16:46:57Z,1.624752e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Stop notebook execution if no new OBIS records were found
if len(new_obis_records) == 0:
    raise SystemExit(
        f"No new OBIS records found for {SCI_NAME}. Stopping notebook execution."
    )

In [ ]:
# Rebuild combined map with new points included

m_combined_with_new = folium.Map(
    location=[
        new_obis_records["decimalLatitude"].mean(),
        new_obis_records["decimalLongitude"].mean()
    ],
    zoom_start=5,
    tiles="OpenStreetMap" #or try "CyclOSM" or "Esri.WorldStreetMap"
)

# Add existing reusable layers
edi_points.add_to(m_combined_with_new)
obis_points.add_to(m_combined_with_new)

# Create reusable layer for new OBIS points
new_obis_points = folium.FeatureGroup(
    name="New OBIS Points",
    show=True
)

# Add new OBIS points to the reusable layer
for _, row in new_obis_records.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).iterrows():

    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in preview_cols
        if col in new_obis_records.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=8,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=1,
        popup=popup,
    ).add_to(new_obis_points)

# Add new-points layer to map
new_obis_points.add_to(m_combined_with_new)

# Zoom to new points
m_combined_with_new.fit_bounds([
    [
        new_obis_records["decimalLatitude"].min(),
        new_obis_records["decimalLongitude"].min()
    ],
    [
        new_obis_records["decimalLatitude"].max(),
        new_obis_records["decimalLongitude"].max()
    ]
])

#show the bounding box used for the full obis query (using pyobis)
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(
        combined_bbox_polygon.wkt,
        max_width=500
    )
).add_to(m_combined_with_new)

# Add layer control after ALL layers are added
folium.LayerControl(collapsed=False).add_to(
    m_combined_with_new
)

m_combined_with_new

In [24]:
# Save Folium map as HTML
#  You can open this html file to view it with any web browser
os.makedirs("../session_files/outputs/", exist_ok=True)
m_combined_with_new.save("../session_files/outputs/session3_combined-data_"+SCI_NAME+".html")

# Wrapping up Session 

Let's get some information we can use in our documentation.

## When did you access your data?

In a notebook cell it's a good idea to print out the date (or datetime) when you ran your code. That way you have it for when you ever need that information in your documentation.

In [25]:
from datetime import datetime

print("Run datetime: " + datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Run datetime: 2026-11-05 19:36:42


## What libarary versions did you use?

Below we are printing the package versions ran with this notebook notebook in case we ever need to look back at this run and know when and what we were working with.  

In [26]:
from importlib.metadata import version

print("\nThis notebook was run with:")

print("pandas version: " + version("pandas"))
print("matplotlib version: " + version("matplotlib"))
print("folium version: " + version("folium"))
print("pyobis version: " + version("pyobis"))
print("shapely version: " + version("shapely"))

# Above we just asked for main package versions, but even better is saving a
#     requirements.txt file which you can write with:
# !pip freeze > requirements.txt
#    will write you a file with ALL versions of everything in your environment


This notebook was run with:
pandas version: 3.0.6
matplotlib version: 3.11.2
folium version: 0.20.0
pyobis version: 1.6.2
shapely version: 2.2.0


## Record your data citations

Record a data citation for any data sources you used.  This is a good practice for what to provide in reports, publications, or journal articles you might publish that used these data.

It's also valuable for the "future you" to have this record in case you want to come back later and work with these data again.

### Citing an EDI dataset

1. Open the dataset landing page. For our CCE-LTER particulate organic carbon and nitrogen dataset, it is https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13
2. Use the suggested citation and DOI shown on that page (**Copy Citation** button). The DOI is specific to this revision (knb-lter-cce.104.13), so it gives the citation for the data we used.

Example:
> California Current Ecosystem LTER and L. Aluwihare. 2025. Particulate organic carbon and nitrogen measurements at selected depths in the water column in the CCE region since 2006 - 2024 (ongoing). ver 13. Environmental Data Initiative. https://doi.org/10.6073/pasta/9feb1de01eb3795e7ea39a8b7ba325e6 (Accessed 2026-10-16).

### Citing OBIS data

OBIS guidance (https://manual.obis.org/citing.html) states: "If you use OBIS data, **you must cite** all datasets you have used, as well as include a general OBIS citation."

#### 1. The general OBIS citation

Use the year you accessed the data:

> OBIS (YEAR) Ocean Biodiversity Information System. Intergovernmental Oceanographic Commission of UNESCO. https://obis.org

#### 2. The source datasets in the OBIS data we used

Our records come from several source datasets. Choose one of these approaches.

**Option A: Cite each dataset individually**

Each source dataset has an OBIS page with its own citation: `https://obis.org/dataset/DATASET_ID`

* The `datasets` table in Part B, Step 3 lists the dataset titles and links for our first query.
* The expanded pyobis query can add datasets not in that table, so check those dataset IDs too.

**Option B: Get one DOI for the data subset you imported**

OBIS can issue one DOI that covers all the source datasets.

1. Run the cell below to print the list of dataset identifiers from our occurrence queries.
2. Go to https://obis.org/doi/ and sign in.
4. Paste the identifiers for all the datasets, and submit the request. OBIS reserves a DOI and records the source datasets in its metadata.
5. Cite the DOI in this form:

> OBIS (YEAR). Custom OBIS export on <date>. Downloaded from: Ocean Biodiversity Information System. Intergovernmental Oceanographic Commission of UNESCO. (Accessed: YYYY-MM-DD). [DOI]



In [27]:
# Get the list of dataset IDs to enter at https://obis.org/doi/ for a citation covering all source OBIS datasets we used

# Unique dataset IDs from the expanded OBIS results (these include the original query area)
ids = sorted(df_obis_expanded["dataset_id"].dropna().unique())

# Print one identifier per line, ready to paste into the DOI request form
print(f"{len(ids)} dataset identifiers:\n")
print("\n".join(ids))

12 dataset identifiers:

04ebe0bf-42cf-4ebe-85ab-bc2ede3c720c
1a653b53-5fe5-4d67-b62c-a69294553304
5b6251f6-a7a5-4dc9-994d-c9504b54776f
71c2c816-7e94-40b9-8e28-8172d9c5fefb
89e23fc8-3f61-4480-9de3-358fe6eefe0b
92af50ad-c9d7-4fe7-b012-eae0e77233a2
a419c8da-35ed-4b62-9709-39b56369c44e
aa16d305-d413-4c4a-90be-b1ec3298d58d
c5687a17-e454-40f9-9a4b-d04b2c812d74
cbd631ea-c8ff-419f-8340-b29c7fdc416a
eaea291a-1e1d-4382-b86f-ac3cc15b8d5a
f180f1e9-817b-49f7-b5f4-04e1cc193f48


Entering those at https://obis.org/doi/ got us a DOI we can now cite for all the subset data we accessed from OBIS.

> OBIS (2026). Distribution records of Pyrosoma in the California Current Large Marine Ecosystem. Downloaded from: Ocean Biodiversity Information System. Intergovernmental Oceanographic Commission of UNESCO. (Accessed: 2026-10-16). https://doi.org/10.25607/obis.export.7bf38bbe

## Check data licenses for your purpose

Summary: Amongst all the data sources we used, the most restrictive are OBIS data that carry a Creative Commons Attribution Non Commercial (CC-BY-NC 4.0) which means the data can not be used for commercial purposes.

This is how you can find licenses for EDI and OBIS data:

EDI:
Take a look at what is in the "Intellectual Rights:" on the EDI page https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13

It's CC0 1.0 (public domain)

OBIS:

If you want to look at the information about each dataset you can prepend "https://obis.org/dataset/" in front of the dataset_id.  
For example in our notebook we loaded data from this dataset:
> https://obis.org/dataset/eaea291a-1e1d-4382-b86f-ac3cc15b8d5a
* which lists the license as CC-BY-NC 4.0

Alternatively the code below uses the OBIS api to show us the license for each dataset we used.

In [28]:
#list all the licenses for datasets we used using the OBIS api
import requests

# Unique dataset IDs from the expanded OBIS results
ids = sorted(df_obis_expanded["dataset_id"].dropna().unique())

# Print the dataset-level license statement (free text) for each dataset
print("dataset_id, dataset-level license (intellectualrights)")
for dataset_id in ids:
    r = requests.get(f"https://api.obis.org/v3/dataset/{dataset_id}", timeout=30) 
    rec = (r.json().get("results") or [{}])[0]
    print(f"{dataset_id}, {str(rec.get('intellectualrights') or 'Not listed')[:150]}")

dataset_id, dataset-level license (intellectualrights)
04ebe0bf-42cf-4ebe-85ab-bc2ede3c720c, To the extent possible under law, the publisher has waived all rights to these data and has dedicated them to the  Public Domain (CC0 1.0)
1a653b53-5fe5-4d67-b62c-a69294553304, This work is licensed under a  Creative Commons Attribution (CC-BY) 4.0 License
5b6251f6-a7a5-4dc9-994d-c9504b54776f, To the extent possible under law, the publisher has waived all rights to these data and has dedicated them to the  Public Domain (CC0 1.0)
71c2c816-7e94-40b9-8e28-8172d9c5fefb, To the extent possible under law, the publisher has waived all rights to these data and has dedicated them to the  Public Domain (CC0 1.0)
89e23fc8-3f61-4480-9de3-358fe6eefe0b, This work is licensed under a  Creative Commons Attribution (CC-BY) 4.0 License
92af50ad-c9d7-4fe7-b012-eae0e77233a2, This work is licensed under a  Creative Commons Attribution (CC-BY) 4.0 License
a419c8da-35ed-4b62-9709-39b56369c44e, Not listed
aa16d305-d4

# Additional Exercise: Suppplemental Dataset - Marine Regions API

Exploring the Marine Regions API

Previously, we made our own geometries and mapped them. However, now we can try pulling geometries from the Marine Regions (marineregions.org) API.

https://www.marineregions.org/gazetteer.php?id=8549&p=details

In [29]:
#At the top of the notebook we defined the region id
#  MRGID = 8549  # California Current (Marine Regions)
#      corresponding to OBIS area id 40003

url = f"https://www.marineregions.org/rest/getGazetteerGeometries.jsonld/{MRGID}/"
r = requests.get(url, timeout=60)
r.raise_for_status()

data = r.json()
geometry_wkt = None

for geom in data.get("mr:hasGeometry", []):
    wkt = geom.get("gsp:asWKT")
    if wkt:
        geometry_wkt = re.sub(r"<[^>]+>\s*", "", wkt).strip()
        break

print("Preview the geom")
print(geometry_wkt[0:100])
print("Geometry length="+str(len(geometry_wkt)))

Preview the geom
POLYGON ((-109.91147613 22.8783226, -108.75467682 21.83747101, -108.7578125 21.83586121, -109.616462
Geometry length=108685


In [33]:
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping
from shapely.validation import make_valid

# Parse the Marine Regions boundary and repair it if invalid
marine_region_geom = shapely_wkt.loads(geometry_wkt)

region_geojson = {
    "type": "Feature",
    "properties": {"MRGID": MRGID},
    "geometry": mapping(marine_region_geom)
}

# Combine the extent of both point datasets and the region boundary
region_minx, region_miny, region_maxx, region_maxy = marine_region_geom.bounds
all_lats = pd.concat([df_edi["Latitude (º)"], df_obis["decimalLatitude"],
                       pd.Series([region_miny, region_maxy])])
all_lons = pd.concat([df_edi["Longitude (º)"], df_obis["decimalLongitude"],
                       pd.Series([region_minx, region_maxx])])

# Build a fresh map with the EDI points, OBIS points, and region boundary
m_final = folium.Map(tiles="OpenStreetMap") #or try "CyclOSM" or "Esri.WorldStreetMap"
 
edi_points.add_to(m_final)
obis_points.add_to(m_final)
new_obis_points.add_to(m_final)

# Add combined bounding box
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(combined_bbox_polygon.wkt, max_width=500)
).add_to(m_final)

#Marine region geometry
folium.GeoJson(
    region_geojson,
    name=f"Marine Region Boundary (MRGID {MRGID})",
    style_function=lambda x: {
        "color": "green",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(f"Marine Region MRGID {MRGID}", max_width=300)
).add_to(m_final)

folium.LayerControl(collapsed=False).add_to(m_final)


# Fit the view to cover all points and the full region boundary
m_final.fit_bounds([[all_lats.min(), all_lons.min()], [all_lats.max(), all_lons.max()]])

m_final

In [31]:
# Save the final map as HTML
#  You can open this html file to view it with any web browser
os.makedirs("../session_files/outputs/", exist_ok=True)
m_final.save("../session_files/outputs/session3_combined-data-"+SCI_NAME + "_with_marine-region.html")

# Additional Exercise: Search for data using EDI's REST API

In the session we found the URL manually on the EDI dataset landing page. But you can find the same data using EDI's rest API with a script.

Documentation for using the EDI Data Package Manager API:
* https://pastaplus-core.readthedocs.io/en/latest/doc_tree/pasta_api/data_package_manager_api.html

Note: Using the API to search requires the same access key we stored in a variable called "myEDIAccessKey" (as we did in in Part A).

In [32]:
import requests
import xml.etree.ElementTree as ET

PASTA = "https://pasta.lternet.edu/package"

# Search CCE-LTER for particulate organic carbon and nitrogen, then keep our dataset of interest (series 104)
r = requests.get(f"{PASTA}/search/eml",
                 params={"q": "particulate organic carbon nitrogen", "fq": "scope:knb-lter-cce",
                         "fl": "packageid,title", "rows": 20, "key": myEDIAccessKey})
hit = next(d for d in ET.fromstring(r.content).findall("document")
           if d.findtext("packageid").startswith("knb-lter-cce.104."))
package_id = hit.findtext("packageid")
scope, identifier, revision = package_id.split(".")

# Get the DOI and the data file (entity) ID for this package
doi = requests.get(f"{PASTA}/doi/eml/{scope}/{identifier}/{revision}", params={"key": myEDIAccessKey}).text
entity_id, entity_name = requests.get(f"{PASTA}/name/eml/{scope}/{identifier}/{revision}",
                                      params={"key": myEDIAccessKey}).text.strip().split(",", 1)

# Build the data URL, which is the base_url used in the Access step
base_url = f"{PASTA}/data/eml/{scope}/{identifier}/{revision}/{entity_id}"

# print info about the dataset we found
print("Title:", hit.findtext("title"))
print("* Landing page:", f"https://portal.edirepository.org/nis/mapbrowse?packageid={package_id}")
print("* DOI:", doi)
print("* Data file:", entity_name)
print("\nThis is the same data we worked with earlier in the session:")
print("Base URL:", base_url)

Title: Particulate organic carbon and nitrogen measurements at selected depths in the water column in the CCE region since 2006 - 2024 (ongoing).
* Landing page: https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13
* DOI: doi:10.6073/pasta/9feb1de01eb3795e7ea39a8b7ba325e6
* Data file: ParticulateOrganicCarbonandNitrogen

This is the same data we worked with earlier in the session:
Base URL: https://pasta.lternet.edu/package/data/eml/knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938
